# Lightweight Network Intrusion Detection System (IDS)
## CA2 Project – Problem Solving for Industry

**Students:**  
- Sander Luiz Santos Soares — **2022164**  
- Thiago Gonçalves da Costa — **2022161**

**College:** CCT College Dublin  
**Programme:** BSc (Hons) in Computing in IT  
**Module:** Problem Solving for Industry  
**Assessment:** CA2 Project  
**Lecturers:** Muhammad Iqbal, Ken Healy  
**Submission Type:** Jupyter Notebook / Python-based IDS prototype  

## Introduction

This project proposes the development of a **Lightweight Network Intrusion Detection System (IDS)** using **Python** and **machine learning** techniques to identify suspicious or potentially malicious network traffic. The main goal is to support the detection of anomalous behaviour in network flows and help organisations identify threats such as DoS, DDoS, brute force activity, port scanning, web attacks, infiltration, and other suspicious patterns.

The proposed solution is designed as a **lightweight and affordable prototype**, especially suitable for **small and medium-sized enterprises (SMEs)**, schools, clinics, small offices, and managed service providers that may not have dedicated cybersecurity teams or advanced monitoring infrastructure. The system aims to reduce the manual effort required to inspect traffic logs and improve the speed of suspicious traffic detection.

This notebook will document the technical development of the project, including **data understanding, preprocessing, feature selection, model training, evaluation, and prototype preparation**, following the logic of the **CRISP-DM framework** required in the CA2 brief. 

The project uses the **CIC-IDS2017** dataset, a public cybersecurity dataset developed by the **Canadian Institute for Cybersecurity**, containing labelled network traffic with both benign and malicious activity. This makes it suitable for building and testing machine learning models for intrusion detection.

The main technologies selected for this project are **Python**, **Scikit-learn**, and **Streamlit**. Python and Scikit-learn will be used for data preprocessing, machine learning model development, and evaluation, while Streamlit will support the creation of a simple dashboard interface for alerts and suspicious traffic summaries.

---

# 1. Data Understanding

## 1.1 Dataset Overview and Merging

The CICIDS2017 dataset is distributed across multiple CSV files, each representing different days of network traffic and specific attack scenarios (e.g., DDoS, PortScan, Web Attacks).

To ensure a comprehensive analysis and avoid bias from individual files, all datasets are merged into a single unified dataset. This allows the model to learn from a complete representation of both benign and malicious traffic patterns.

In [6]:
import pandas as pd
import glob

# Using glob to load all CSV files within the folder
files = glob.glob("Datasets/CICIDS2017/*.csv")


# Read and combine into one dataset
df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

#notice some column have a space before, so using this to remove unwanted spaces.
df.columns = df.columns.str.strip()

print("Number of files loaded:", len(files))
print("Dataset shape:", df.shape)

Number of files loaded: 8
Dataset shape: (2830743, 79)


---

## 1.2 Initial Data Exploration

After merging the datasets, an initial exploration is performed to understand the structure of the data. 
This includes examining the first rows, dataset dimensions, feature names, and data types.

In [2]:
# View first rows
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
# Dataset shape
df.shape

(2830743, 79)

In [4]:
# Column names
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [5]:
# Data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

---

## 1.3 Target Variable Analysis

The target variable in this dataset is the 'Label' column, which indicates whether a network flow is benign or corresponds to a specific type of attack.

To better understand the dataset, both the absolute count and percentage distribution of each class are analysed. This helps identify class imbalance, which is common in intrusion detection datasets and can impact model performance.

In [10]:
# Count values of each class
class_counts = df['Label'].value_counts()

# Percentage distribution
class_percentages = df['Label'].value_counts(normalize=True) * 100

# Combine into one table
class_distribution = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_percentages
})

class_distribution

,Count,Percentage (%)
Label,,
BENIGN,2273097,80.300366
DoS Hulk,231073,8.162981
PortScan,158930,5.614427
DDoS,128027,4.522735
DoS GoldenEye,10293,0.363615
FTP-Patator,7938,0.280421
SSH-Patator,5897,0.208320
DoS slowloris,5796,0.204752
DoS Slowhttptest,5499,0.194260
